# Transfer Learning CIFAR10

* Train a simple convnet on the CIFAR dataset the first 5 output classes [0..4].
* Freeze convolutional layers and fine-tune dense layers for the last 5 ouput classes [5..9].


In [1]:
import tensorflow as tf
import keras

import numpy as np
import pandas as pd
from keras.datasets import mnist
from keras.utils import np_utils
from sklearn import metrics
import matplotlib.pyplot as plt
%matplotlib inline

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.layers import Conv2D,MaxPooling2D
from keras.constraints import maxnorm

C:\ProgramData\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:526: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
C:\ProgramData\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:527: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
C:\ProgramData\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:528: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
C:\ProgramData\Anaconda3\lib\site-packages\tensorflow\python\framework\dtypes.py:529: FutureWarning: Passi

### 1. Import CIFAR10 data and create 2 datasets with one dataset having classes from 0 to 4 and other having classes from 5 to 9 

In [2]:
from keras.datasets import cifar10

In [3]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [4]:
X_train_0_4 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
X_train_5_9 = np.asarray([X_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

X_test_0_4 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
X_test_5_9 = np.asarray([X_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [5]:
Y_train_0_4 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 0 and int(label) <= 4])
Y_train_5_9 = np.asarray([y_train[key] for (key, label) in enumerate(y_train) if int(label) >= 5 and int(label) <= 9])

Y_test_0_4 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 0 and int(label) <= 4])
Y_test_5_9 = np.asarray([y_test[key] for (key, label) in enumerate(y_test) if int(label) >= 5 and int(label) <= 9])

In [6]:
print('X_train_0_4:', X_train_0_4.shape)
print('Y_train_0_4:', Y_train_0_4.shape)
print('X_test_0_4:', X_test_0_4.shape)
print('Y_test_0_4:', Y_test_0_4.shape)
print('X_train_5_9:', X_train_5_9.shape)
print('Y_train_5_9:', Y_train_5_9.shape)
print('X_test_5_9:', X_test_5_9.shape)
print('Y_test_5_9:', Y_test_5_9.shape)

X_train_0_4: (25000, 32, 32, 3)
Y_train_0_4: (25000, 1)
X_test_0_4: (5000, 32, 32, 3)
Y_test_0_4: (5000, 1)
X_train_5_9: (25000, 32, 32, 3)
Y_train_5_9: (25000, 1)
X_test_5_9: (5000, 32, 32, 3)
Y_test_5_9: (5000, 1)


In [7]:
np.unique(Y_train_0_4)

array([0, 1, 2, 3, 4], dtype=uint8)

In [8]:
np.unique(Y_train_5_9)

array([5, 6, 7, 8, 9], dtype=uint8)

In [9]:
np.unique(Y_test_0_4)

array([0, 1, 2, 3, 4])

In [10]:
np.unique(Y_test_5_9)

array([5, 6, 7, 8, 9])

### 2. Use One-hot encoding to divide y_train and y_test into required no of output classes

In [11]:
num_class_04 = len(np.unique(Y_train_0_4))

Y_train_0_4 = tf.keras.utils.to_categorical(Y_train_0_4, num_classes=num_class_04)
Y_test_0_4 = tf.keras.utils.to_categorical(Y_test_0_4, num_classes=num_class_04)

### 3. Build a sequential neural network model which can classify the classes 0 to 4 of CIFAR10 dataset with at least 80% accuracy on test data

In [12]:
# Making sure that the values are float so that we can get decimal points after division
X_train_0_4 = X_train_0_4.astype('float32')
X_test_0_4 = X_test_0_4.astype('float32')

# Normalizing the RGB codes by dividing it to the max RGB value.
X_train_0_4 = X_train_0_4 / 255.0
X_test_0_4 = X_test_0_4 / 255.0

In [13]:
print('X_train_0_4:', X_train_0_4.shape)
print('Y_train_0_4:', Y_train_0_4.shape)
print('X_train_5_9:', X_train_5_9.shape)
print('Y_train_5_9:', Y_train_5_9.shape)
print('X_test_0_4:', X_test_0_4.shape)
print('Y_test_0_4:', Y_test_0_4.shape)
print('X_test_5_9:', X_test_5_9.shape)
print('Y_test_5_9:', Y_test_5_9.shape)

X_train_0_4: (25000, 32, 32, 3)
Y_train_0_4: (25000, 5)
X_train_5_9: (25000, 32, 32, 3)
Y_train_5_9: (25000, 1)
X_test_0_4: (5000, 32, 32, 3)
Y_test_0_4: (5000, 5)
X_test_5_9: (5000, 32, 32, 3)
Y_test_5_9: (5000, 1)


In [25]:
# initialize model
model = Sequential()

# add input layer with input_size(32,32,3)
model.add(Conv2D(32, (3, 3), input_shape=(32, 32, 3), activation='relu'))

# create Conv 2D layers , along with DropoUts and MaxPooling to avoid overfit
model.add(Dropout(0.2))
model.add(Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(Dropout(0.2))
model.add(Conv2D(128, (3, 3), activation='relu', padding='same'))
model.add(MaxPooling2D())
# Flatten output
model.add(Flatten())
model.add(Dropout(0.2))
# add Dense layers
model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.3))

# add Output layer
model.add(Dense(num_class_04, activation='softmax'))

In [26]:
# Compile model
from keras.optimizers import SGD

epochs = 30
lrate = 0.01
decay = lrate/epochs

sgd = tf.keras.optimizers.SGD(lr=lrate, momentum=0.9, decay=decay, nesterov=False)
model.compile(loss='categorical_crossentropy', optimizer=sgd, metrics=['accuracy'])
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_8 (Conv2D)            (None, 30, 30, 32)        896       
_________________________________________________________________
dropout_7 (Dropout)          (None, 30, 30, 32)        0         
_________________________________________________________________
conv2d_9 (Conv2D)            (None, 30, 30, 32)        9248      
_________________________________________________________________
max_pooling2d_4 (MaxPooling2 (None, 15, 15, 32)        0         
_________________________________________________________________
conv2d_10 (Conv2D)           (None, 15, 15, 64)        18496     
_________________________________________________________________
dropout_8 (Dropout)          (None, 15, 15, 64)        0         
_________________________________________________________________
conv2d_11 (Conv2D)           (None, 15, 15, 64)        36928     
__________

In [27]:
# Fit the model
model.fit(X_train_0_4, Y_train_0_4, validation_data=(X_test_0_4, Y_test_0_4), epochs=epochs, batch_size=64)

Train on 25000 samples, validate on 5000 samples
Epoch 1/30
25000/25000 [==============================] - 246s 10ms/sample - loss: 1.4447 - acc: 0.3704 - val_loss: 1.3498 - val_acc: 0.4286
Epoch 2/30
25000/25000 [==============================] - 262s 10ms/sample - loss: 1.1758 - acc: 0.5096 - val_loss: 1.0247 - val_acc: 0.5820
Epoch 3/30
25000/25000 [==============================] - 272s 11ms/sample - loss: 0.9990 - acc: 0.5992 - val_loss: 0.9477 - val_acc: 0.6216
Epoch 4/30
25000/25000 [==============================] - 274s 11ms/sample - loss: 0.9018 - acc: 0.6370 - val_loss: 0.9170 - val_acc: 0.6404
Epoch 5/30
25000/25000 [==============================] - 244s 10ms/sample - loss: 0.8457 - acc: 0.6668 - val_loss: 0.7896 - val_acc: 0.6960
Epoch 6/30
25000/25000 [==============================] - 245s 10ms/sample - loss: 0.7915 - acc: 0.6925 - val_loss: 0.8042 - val_acc: 0.6868
Epoch 7/30
25000/25000 [==============================] - 252s 10ms/sample - loss: 0.7530 - acc: 0.7032 -

In [28]:
score = model.evaluate(X_test_0_4, Y_test_0_4)
print('\n', 'Test accuracy:', score[1]*100)

5000/5000 [==============================] - 13s 3ms/sample - loss: 0.4923 - acc: 0.8284

 Test accuracy: 82.84000158309937


### 4. In the model which was built above (for classification of classes 0-4 in CIFAR10), make only the dense layers to be trainable and conv layers to be non-trainable

In [29]:
#Set pre-trained  layers to be non trainable
for layer in model.layers[:-5]:
    layer.trainable = False

In [30]:
for layer in model.layers:
    print(layer, layer.trainable)

<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D1319FA0B8> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000002D1187EC278> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D1319FA3C8> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x000002D1319FABE0> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D1319FAB00> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000002D131E29F28> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D131E23CF8> False
<tensorflow.python.keras.layers.pooling.MaxPooling2D object at 0x000002D131E23588> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D131E85D68> False
<tensorflow.python.keras.layers.core.Dropout object at 0x000002D131F00320> False
<tensorflow.python.keras.layers.convolutional.Conv2D object at 0x000002D131F11F60> False
<tensorflow.python.keras.layers.pooling.MaxPo

### 5. Utilize the the model trained on CIFAR 10 (classes 0 to 4) to classify the classes 5 to 9 of CIFAR 10  (Use Transfer Learning) <br>
Achieve an accuracy of more than 85% on test data

In [31]:
# Pre-process labels as one hot encoding doesn't start it's label with 5
Y_train_5_9 = Y_train_5_9 - 5
Y_test_5_9 = Y_test_5_9 - 5

In [32]:
# Do one hot encoding
num_class_59 = len(np.unique(Y_train_5_9))

Y_train_5_9 = tf.keras.utils.to_categorical(Y_train_5_9, num_classes=num_class_59)
Y_test_5_9 = tf.keras.utils.to_categorical(Y_test_5_9, num_classes=num_class_59)

IndexError: index -5 is out of bounds for axis 1 with size 2

In [ ]:
# normalize input
X_train_5_9 = X_train_5_9.astype("float32")/255.0
X_test_5_9 = X_test_5_9.astype("float32")/255.0

In [ ]:
# fit the model by retraining only dense layers 

epochs=10
batch_size = 64

model.fit(X_train_5_9, Y_train_5_9, validation_data=(X_test_5_9, Y_test_5_9), batch_size=batch_size, epochs=epochs)

In [ ]:
score = model.evaluate(X_test_59, y_test_59)
print('\n', 'Test accuracy for Model with Transfer Learning :', score[1]*100)

# Text classification using TF-IDF

### 6. Load the dataset from sklearn.datasets

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from pprint import pprint

### 7. Training data

In [ ]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

In [ ]:
train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

In [ ]:
pprint(list(twenty_train.target_names))

### 8. Test data

In [ ]:
test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

###  a.  You can access the values for the target variable using .target attribute 

In [ ]:
train.target


###  b. You can access the name of the class in the target variable with .target_names


In [ ]:
train.target_names

In [ ]:
train.data[0:2]

### 9.  Now with dependent and independent data available for both train and test datasets, using TfidfVectorizer fit and transform the training data and test data and get the tfidf features for both

In [ ]:
# use TfIDFVectorizer to create document-term matrices 
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer()

In [ ]:
train_dtm = vect.fit_transform(train.data)

In [ ]:
test_dtm = vect.transform(test.data)

### 10. Use logisticRegression with tfidf features as input and targets as output and train the model and report the train and test accuracy score

In [ ]:
# import logistic regression model
from sklearn.linear_model import LogisticRegression

# instantiate the model
lr = LogisticRegression()

In [ ]:
# train the model 
lr.fit(train_dtm, train.target)

In [ ]:
# make class predictions
yPred = lr.predict(test_dtm)

In [ ]:
# calculate accuracy
from sklearn import metrics

acc=metrics.accuracy_score(test.target, yPred)*100
acc